# [5.5] Diffusion Language Models - Exercises

Implement a toy discrete diffusion LM stack: mask schedules, forward noising, masked denoising loss, confidence remasking, iterative sampling, commitment-time diagnostics, and a tiny bidirectional denoiser.

```yaml
gt_tier: GT-0
exercise_id: 5.5-diffusion-language-models
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA tiny-model preflight
requires_gpu: true for the trained tiny diffusion LM; false for the implementation exercises
```

Reading map: review masked-language-model loss and transformer encoder blocks. Failure modes to watch for: loss leakage through unmasked positions, sampling low-to-high instead of high-to-low noise, keeping stale token values during remasking, and treating this tiny path as DiffusionGemma checkpoint parity.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn as nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part5_diffusion_language_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_diffusion_language_models.tests as tests


@dataclass(frozen=True)
class DiscreteDiffusionSchedule:
    mask_probs: t.Tensor
    mask_token_id: int

    @property
    def num_steps(self) -> int:
        return int(self.mask_probs.numel())


@dataclass(frozen=True)
class NoisingResult:
    noisy_tokens: t.Tensor
    mask: t.Tensor
    timesteps: t.Tensor


@dataclass(frozen=True)
class DenoisingStepStats:
    step: int
    mask_fraction: float
    mean_entropy: float
    committed_fraction: float


## Mask Schedule

Difficulty: easy. Importance: high. Expected output: the linear schedule and expected fraction tests should pass. Common bug: storing the schedule in sampling order rather than low-noise to high-noise order.


In [ ]:
def linear_mask_schedule(
    num_steps: int,
    *,
    mask_token_id: int,
    min_mask_prob: float = 0.0,
    max_mask_prob: float = 1.0,
) -> DiscreteDiffusionSchedule:
    raise NotImplementedError()


def expected_mask_fraction(schedule: DiscreteDiffusionSchedule, timesteps: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_linear_mask_schedule_and_expected_fraction(
    linear_mask_schedule,
    expected_mask_fraction,
)


## Forward Noising

Difficulty: easy. Importance: high. Expected output: timestep 0 should leave tokens unchanged, and the max-noise timestep should mask every token. Common bug: sampling one mask decision per sequence instead of per token.


In [ ]:
def apply_forward_noising(
    input_ids: t.Tensor,
    timesteps: t.Tensor,
    schedule: DiscreteDiffusionSchedule,
    *,
    generator: t.Generator | None = None,
) -> NoisingResult:
    raise NotImplementedError()


tests.test_forward_noising_extremes_and_seeded_masks(
    apply_forward_noising,
    linear_mask_schedule,
)


## Masked Denoising Loss

Difficulty: medium. Importance: high. Expected output: confident masked-token logits should have low loss even if an unmasked position has a wrong high logit. Common bug: averaging loss over every token.


In [ ]:
def masked_denoising_loss(logits: t.Tensor, target_ids: t.Tensor, mask: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_masked_denoising_loss_uses_only_masked_positions(masked_denoising_loss)


## Remasking

Difficulty: medium. Importance: high. Expected output: confidence remasking should mask the requested number of low-confidence positions, and entropy should match categorical entropy. Common bug: preserving old token values instead of filling from the current prediction.


In [ ]:
def token_entropy(logits: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def confidence_remask(
    logits: t.Tensor,
    current_tokens: t.Tensor,
    *,
    mask_token_id: int,
    next_mask_fraction: float,
) -> t.Tensor:
    raise NotImplementedError()


def uniform_remask(
    tokens: t.Tensor,
    *,
    mask_token_id: int,
    next_mask_fraction: float,
    generator: t.Generator | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_confidence_remask_entropy_and_uniform_control(
    confidence_remask,
    token_entropy,
    uniform_remask,
)


## Oracle Sampler

Difficulty: medium. Importance: high. Expected output: an oracle denoiser should reconstruct the target exactly and report one stats row per denoising step. Common bug: iterating low-to-high noise instead of high-to-low.


In [ ]:
def diffusion_sampler(
    model_fn,
    *,
    shape: tuple[int, int],
    schedule: DiscreteDiffusionSchedule,
    temperature: float = 0.0,
    remask: str = "confidence",
    generator: t.Generator | None = None,
    device: t.device | None = None,
) -> tuple[t.Tensor, list[DenoisingStepStats]]:
    raise NotImplementedError()


tests.test_oracle_diffusion_sampler_recovers_target(
    diffusion_sampler,
    linear_mask_schedule,
)


## Diagnostics

Difficulty: medium. Importance: high. Expected output: commitment times, edit distance, and activation trajectory validation should match the worked examples. Common bug: confusing denoising step numbers with stored trajectory indices.


In [ ]:
def commitment_times(tokens_over_steps: t.Tensor, mask_token_id: int) -> t.Tensor:
    raise NotImplementedError()


def edit_distance(a: list[int], b: list[int]) -> int:
    raise NotImplementedError()


def validate_activation_trajectory(
    activations: list[t.Tensor],
    *,
    expected_steps: int,
    batch: int,
    seq_len: int,
) -> bool:
    raise NotImplementedError()


tests.test_commitment_edit_distance_and_activation_trajectory(
    commitment_times,
    edit_distance,
    validate_activation_trajectory,
)


## Tiny Diffusion Denoiser

Difficulty: medium. Importance: medium. Expected output: the wrapper should return `(batch, seq, vocab)` logits. Common bug: using a causal decoder mask; this toy diffusion denoiser is bidirectional.


In [ ]:
class TinyConditionalDiffusionLM(nn.Module):
    def __init__(
        self,
        *,
        vocab_size: int = 11,
        seq_len: int = 6,
        num_steps: int = 6,
        d_model: int = 96,
    ) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.position_embed = nn.Embedding(seq_len, d_model)
        self.time_embed = nn.Embedding(num_steps, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=2 * d_model,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.unembed = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: t.Tensor, timesteps: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


tests.test_tiny_conditional_diffusion_lm_forward_shape(TinyConditionalDiffusionLM)


## Verification Block

After completing the exercises, compare with `solutions.py`. The CUDA path trains a tiny conditional denoiser on generated copy-pair data and checks held-out accuracy, sampler exact match, shuffled-label controls, activation trajectory shapes, and VRAM. It also reports the released DiffusionGemma state: Google BF16 direct local loading is deferred for the 24GB tier, while the pinned NVIDIA NVFP4 checkpoint has a real isolated vLLM generation proof. This is still not full DiffusionGemma interpretability parity.


In [ ]:
# Uncomment after checking your implementations against the section solution.
# from part5_diffusion_language_models.solutions import run_smoke_test, run_gpu_test
# run_smoke_test(cpu=True)
# run_gpu_test(max_vram_gb=24.0)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate. The tiny-model preflight and pinned NVFP4 generation proof can pass while diffusion-time activation capture and patching work remain; the summary below shows that split explicitly.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu.get(key) for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
    "diffusiongemma_generation_ready",
    "diffusiongemma_nvfp4_isolated_vllm_generation_ready",
    "diffusiongemma_external_vllm_torch_version",
    "diffusiongemma_external_vllm_vllm_version",
    "diffusiongemma_vllm_probe_output_nonempty",
    "diffusiongemma_nvfp4_local_ready_for_vllm",
    "diffusiongemma_nvfp4_quant_method",
    "diffusiongemma_nvfp4_transformers_quantization_supported",
    "diffusiongemma_vllm_preserves_current_torch_cuda_stack",
    "diffusiongemma_blockers",
] if key in gpu}
